# Web Scraping

## Objetivo

Crear un scraper en Python capaz de obtener productos desde un
sitio web, localizar los enlaces de las vistas de detalle y
extraer información relevante de cada producto.

Para este ejercicio se utilizó **Books to Scrape**, un sitio
diseñado para practicar técnicas de Web Scraping.

Sitio utilizado:

https://books.toscrape.com/


## 1. Librerías utilizadas

Se utilizaron las siguientes librerías:

- `requests`: realizar peticiones HTTP.
- `BeautifulSoup`: analizar el contenido HTML.
- `urljoin`: construir correctamente las URLs de los productos.


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

## 2. Vista de listado

La primera página del catálogo contiene 20 productos.

Para localizar los enlaces que llevan a la vista de detalle
de cada producto se utilizó el selector CSS:

`article.product_pod h3 a`

Este selector encuentra el enlace contenido dentro del título
de cada producto.


In [ ]:
BASE_URL = "https://books.toscrape.com/"

LIST_URL = urljoin(
    BASE_URL,
    "catalogue/page-1.html",
)


def obtener_links_productos():
    response = requests.get(
        LIST_URL,
        timeout=10,
    )
    response.raise_for_status()

    soup = BeautifulSoup(
        response.content,
        "html.parser",
    )

    productos = soup.select(
        "article.product_pod h3 a"
    )

    links = []

    for producto in productos:
        link = producto.get("href")

        url_completa = urljoin(
            LIST_URL,
            link,
        )

        links.append(url_completa)

    return links

## 3. Selectores de la vista de detalle

Después de entrar a la página individual de cada producto se
utilizaron los siguientes selectores:

- Título: `div.product_main h1`
- Precio: `p.price_color`
- Disponibilidad: `p.instock.availability`
- Categoría: `ul.breadcrumb li a`
- Calificación: `p.star-rating`
- Información del producto: `table.table.table-striped tr`

La tabla de información se recorre hasta localizar la fila
correspondiente al `UPC`.


In [ ]:
def obtener_detalle_producto(url):
    response = requests.get(
        url,
        timeout=10,
    )
    response.raise_for_status()

    soup = BeautifulSoup(
        response.content,
        "html.parser",
    )

    titulo = soup.select_one(
        "div.product_main h1"
    ).get_text(strip=True)

    precio = soup.select_one(
        "p.price_color"
    ).get_text(strip=True)

    precio = precio.replace("Â", "")

    disponibilidad = soup.select_one(
        "p.instock.availability"
    ).get_text(
        " ",
        strip=True,
    )

    categoria = soup.select(
        "ul.breadcrumb li a"
    )[-1].get_text(strip=True)

    rating_element = soup.select_one(
        "p.star-rating"
    )

    rating = rating_element.get(
        "class"
    )[-1]

    filas = soup.select(
        "table.table.table-striped tr"
    )

    upc = ""

    for fila in filas:
        encabezado = fila.select_one("th")
        valor = fila.select_one("td")

        if (
            encabezado
            and valor
            and encabezado.get_text(strip=True)
            == "UPC"
        ):
            upc = valor.get_text(strip=True)
            break

    return {
        "titulo": titulo,
        "precio": precio,
        "disponibilidad": disponibilidad,
        "categoria": categoria,
        "rating": rating,
        "upc": upc,
    }

## 4. Ejecución del scraper

Para comprobar el funcionamiento se obtienen todos los enlaces
de la primera página y posteriormente se extrae la información
de los primeros cinco productos.


In [ ]:
links = obtener_links_productos()

print(
    f"Productos encontrados: {len(links)}"
)

for numero, link in enumerate(
    links[:5],
    start=1,
):
    producto = obtener_detalle_producto(
        link
    )

    print(f"\nProducto {numero}")
    print(
        f"Título: {producto['titulo']}"
    )
    print(
        f"Precio: {producto['precio']}"
    )
    print(
        "Disponibilidad: "
        f"{producto['disponibilidad']}"
    )
    print(
        f"Categoría: {producto['categoria']}"
    )
    print(
        f"Rating: {producto['rating']}"
    )
    print(
        f"UPC: {producto['upc']}"
    )
    print(f"URL: {link}")

## 5. Resultados obtenidos

El scraper encontró correctamente **20 productos** en la
primera página del catálogo.

Los primeros cinco productos obtenidos fueron:

1. **A Light in the Attic**
   - Precio: £51.77
   - Disponibilidad: In stock (22 available)
   - Categoría: Poetry
   - Rating: Three
   - UPC: a897fe39b1053632

2. **Tipping the Velvet**
   - Precio: £53.74
   - Disponibilidad: In stock (20 available)
   - Categoría: Historical Fiction
   - Rating: One
   - UPC: 90fa61229261140a

3. **Soumission**
   - Precio: £50.10
   - Disponibilidad: In stock (20 available)
   - Categoría: Fiction
   - Rating: One
   - UPC: 6957f44c3847a760

4. **Sharp Objects**
   - Precio: £47.82
   - Disponibilidad: In stock (20 available)
   - Categoría: Mystery
   - Rating: Four
   - UPC: e00eb4fd7b871a48

5. **Sapiens: A Brief History of Humankind**
   - Precio: £54.23
   - Disponibilidad: In stock (20 available)
   - Categoría: History
   - Rating: Five
   - UPC: 4165285e1663650f


## Conclusión

El scraper logró localizar los enlaces de los productos desde
la vista de listado y posteriormente acceder a cada vista de
detalle.

Mediante selectores CSS y BeautifulSoup se extrajeron
correctamente el título, precio, disponibilidad, categoría,
calificación y UPC de los productos.

Las pruebas realizadas confirmaron que el scraper puede
recuperar correctamente la información del catálogo.
